<a href="https://colab.research.google.com/github/Dado233242/intro-ai-/blob/main/lab2_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №2 Задача классификации

## Задание
Решите пожалуйста следующую задачу.

Используйте классификатор SVM для данных находящихся пo
[ссылке](https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link) (данный датасет применялся в лабораторной работе №1). Здесь - выходящей меткой (учителем) является столбец / атрибут - `outcome`. Предоставьте результаты, используя следующие ядерные функции:
- Линейное ядро
- Полиномиальное ядро
- Гауссово ядро
- Сигмоидное ядро

Укажите `score` моделей. Сопоставьте полученные результаты и сделайте вывод.

## Загрузка данных

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [16]:
! gdown --fuzzy 'https://drive.google.com/file/d/1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec/view?usp=share_link'

Downloading...
From: https://drive.google.com/uc?id=1LBDnhITL0Wqwp5G6M6IBI-SSz8BIoNec
To: /content/diabetes.csv
100% 23.1k/23.1k [00:00<00:00, 38.1MB/s]


In [17]:
df = pd.read_csv('/content/diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


**Замечание о данных.** В столбцах Glucose, BloodPressure, SkinThickness, Insulin и BMI встречаются нулевые значения, которые физиологически невозможны и, по-видимому, обозначают пропущенные измерения (например, у Insulin и SkinThickness их особенно много). В данной работе эти значения оставлены без изменений, чтобы не отклоняться от условия задания; при дальнейшем исследовании их следует обработать (например, заменить на NaN и импутировать медианой, вычисленной по обучающей выборке).

##  О датасете
[Pima Indians Diabetes Database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database/data)

Этот набор данных взят из Национального института диабета, заболеваний органов пищеварения и почек (NIDDK). Целью набора данных является диагностическое предсказание наличия у пациента диабета на основе определенных диагностических измерений, включенных в набор данных. На выбор этих экземпляров из более крупной базы данных было наложено несколько ограничений. В частности, все пациенты здесь — женщины не моложе 21 года, родом из индейцев Пима.

Наборы данных состоят из нескольких переменных медицинских предикторов и одной целевой переменной `Outcome`. Переменные-предикторы включают количество беременностей, которые были у пациентки, её индекс массы тела `BMI`, уровень инсулина, возраст и так далее. Всего в датасете 9 переменных:
- `Pregnancies` - количество беременностей;
- `Glucose` - концентрация глюкозы в плазме через 2 часа при пероральном тесте на толерантность к глюкозе;
- `BloodPressure` - диастолическое артериальное давление (мм рт. ст.);
- `SkinThickness` - толщина кожной складки трицепса (мм);
- `Insulin` - 2-часовой сывороточный инсулин (мкЕд/мл);
- `BMI` - Индекс массы тела (вес в кг/(рост в м)^2);
- `DiabetesPedigreeFunction` - функция оценки вероятности диабета на основе семейного анамнеза;
- `Age` - возраст (лет);
- `Outcome` - целевая переменная (0 или 1) 268 из 768 — 1 (больны), остальные — 0 (здоровы).



In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [19]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [20]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

## Классификатор SVM

In [21]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Масштабирование данных

In [22]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Линейное ядро

In [23]:
# Обучение модели с линейным ядром
linear_svm = SVC(kernel='linear', C=1.0, random_state=42)
linear_svm.fit(X_train_scaled, y_train)
linear_score = accuracy_score(y_test, linear_svm.predict(X_test_scaled))
print(f"Линейное ядро Score: {linear_score:.4f}")

Линейное ядро Score: 0.7208


### Полиномиальное ядро

In [24]:
poly_svm = SVC(kernel='poly', degree=3, C=1.0, random_state=42)
poly_svm.fit(X_train_scaled, y_train)
poly_score = accuracy_score(y_test, poly_svm.predict(X_test_scaled))
print(f"Полиномиальное ядро Score: {poly_score:.10f}")

Полиномиальное ядро Score: 0.7532467532


### Гауссово ядро

In [25]:
rbf_svm = SVC(kernel='rbf', C=1.0, random_state=42)
rbf_svm.fit(X_train_scaled, y_train)
rbf_score = accuracy_score(y_test, rbf_svm.predict(X_test_scaled))
print(f"Гауссово ядро Score: {rbf_score:.10f}")

Гауссово ядро Score: 0.7532467532


### Сигмоидное ядро

In [26]:
sigmoid_svm = SVC(kernel='sigmoid', C=1.0, random_state=42)
sigmoid_svm.fit(X_train_scaled, y_train)
sigmoid_score = accuracy_score(y_test, sigmoid_svm.predict(X_test_scaled))
print(f"Сигмоидное ядро Score: {sigmoid_score:.4f}")

Сигмоидное ядро Score: 0.7078


## Вывод

Сравнены SVC с четырьмя ядрами (linear, poly со степенью 3, RBF и sigmoid; C=1.0) на одном стратифицированном разбиении 80/20 (random_state=42). Признаки стандартизованы по обучающей выборке. Оба класса Outcome сохранены: строки не исключались.

Accuracy на тестовой выборке:
- линейное ядро — 72,08 %;
- полиномиальное ядро — 75,32 %;
- гауссово ядро (RBF) — 75,32 %;
- сигмоидное ядро — 70,78 %.

Наибольшая accuracy получена для полиномиального и гауссова ядер, наименьшая — для сигмоидного. Линейное и сигмоидное ядра дают разные результаты. Одинаковая accuracy у poly и RBF не означает, что модели дают одинаковые прогнозы: они могут ошибаться на разных объектах.

Результаты относятся к выбранному разбиению и фиксированным параметрам, поэтому нельзя делать вывод о превосходстве какого-либо ядра в общем случае.